# Day 57 · Exercise 3: Deployment File Generators

**What you'll build:** Implement `generate_procfile`, `generate_requirements`, and `write_deploy_files` — the functions that create the deployment manifest files Render, Railway, and Heroku read to know how to start and install your app.

## Setup (provided)

In [ ]:
from pathlib import Path


## Your Implementation

In [ ]:
def generate_procfile(start_command: str) -> str:
    """Generate the contents of a Procfile for Render/Railway/Heroku.

    Args:
        start_command: The command to start the web server,
                       e.g. "uvicorn deploy_api:app --host 0.0.0.0 --port $PORT"
    Returns:
        A string with the format "web: <start_command>\n"
    """
    # TODO: return f"web: {start_command}\n"
    raise NotImplementedError

def generate_requirements(packages: list[str]) -> str:
    """Generate a requirements.txt file contents string.

    Args:
        packages: List of package specifiers (e.g. ["fastapi>=0.100", "uvicorn"]).
    Returns:
        A newline-joined string of packages sorted alphabetically,
        with a trailing newline.
    """
    # TODO: return "\n".join(sorted(packages)) + "\n"
    raise NotImplementedError

def write_deploy_files(directory: str, start_command: str,
                       packages: list[str]) -> dict[str, str]:
    """Write Procfile and requirements.txt into directory. Return paths dict.

    Args:
        directory:     Path to the output directory (created if missing).
        start_command: Passed to generate_procfile.
        packages:      Passed to generate_requirements.
    Returns:
        {"procfile": <path>, "requirements": <path>}
    """
    # TODO:
    # 1. Path(directory).mkdir(parents=True, exist_ok=True)
    # 2. Write Procfile: Path(directory) / "Procfile"
    # 3. Write requirements.txt: Path(directory) / "requirements.txt"
    # 4. Return dict with string paths
    raise NotImplementedError


In [ ]:
def generate_procfile(start_command: str) -> str:
    return f"web: {start_command}\n"

def generate_requirements(packages: list[str]) -> str:
    return "\n".join(sorted(packages)) + "\n"

def write_deploy_files(directory: str, start_command: str,
                       packages: list[str]) -> dict[str, str]:
    d = Path(directory)
    d.mkdir(parents=True, exist_ok=True)
    procfile = d / "Procfile"
    reqs     = d / "requirements.txt"
    procfile.write_text(generate_procfile(start_command), encoding="utf-8")
    reqs.write_text(generate_requirements(packages), encoding="utf-8")
    return {"procfile": str(procfile), "requirements": str(reqs)}


## Check Your Work

In [ ]:
def _run_checks():
    import tempfile
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        pf = generate_procfile("uvicorn app:app --host 0.0.0.0 --port $PORT")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: generate_procfile not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, pf.startswith("web: uvicorn"),
         f"Procfile starts with 'web:' (got {pf!r:.60})")
    _chk(2, pf.endswith("\n"),
         f"Procfile ends with newline (got {pf!r:.60})")

    try:
        pkgs = ["fastapi>=0.100", "uvicorn", "httpx"]
        reqs = generate_requirements(pkgs)
    except NotImplementedError:
        for i in range(3, 5):
            print(f"  ❌ Check {i}: generate_requirements not implemented")
        reqs = None

    if reqs is not None:
        lines = [l for l in reqs.split("\n") if l]
        _chk(3, lines == sorted(pkgs),
             f"requirements sorted alphabetically (got {lines})")
        _chk(4, reqs.endswith("\n"),
             f"requirements.txt ends with newline")
    else:
        for i in range(3, 5):
            print(f"  ❌ Check {i}: skipped")

    try:
        with tempfile.TemporaryDirectory() as td:
            result = write_deploy_files(td, "uvicorn app:app --port $PORT",
                                        ["fastapi", "uvicorn"])
            pf_path = Path(result["procfile"])
            rq_path = Path(result["requirements"])
            _chk(5, pf_path.exists() and rq_path.exists() and
                 pf_path.name == "Procfile" and rq_path.name == "requirements.txt",
                 f"files written: {pf_path.name}, {rq_path.name}")
    except NotImplementedError:
        print(f"  ❌ Check 5: write_deploy_files not implemented")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Add a `generate_render_yaml(service_name, start_command, env_vars)` function that produces a `render.yaml` service manifest. The output should be a YAML string with `services:` → one web service entry with the given name, startCommand, and envVars list. You can build it as an f-string without importing PyYAML — the output is small and predictable.

## Solution

<details>
<summary>Show solution</summary>

```python
def generate_procfile(start_command: str) -> str:
    return f"web: {start_command}\n"

def generate_requirements(packages: list[str]) -> str:
    return "\n".join(sorted(packages)) + "\n"

def write_deploy_files(directory: str, start_command: str,
                       packages: list[str]) -> dict[str, str]:
    d = Path(directory)
    d.mkdir(parents=True, exist_ok=True)
    procfile = d / "Procfile"
    reqs     = d / "requirements.txt"
    procfile.write_text(generate_procfile(start_command), encoding="utf-8")
    reqs.write_text(generate_requirements(packages), encoding="utf-8")
    return {"procfile": str(procfile), "requirements": str(reqs)}
```

**Why this works:** A `Procfile` is the simplest possible deployment manifest —
one line per process type. `web: <command>` tells the platform what command
to run for HTTP traffic. `$PORT` is set by the platform at runtime (never
hardcode the port). Sorting packages in `requirements.txt` is a best practice
for readable diffs. `write_deploy_files` wraps both generators and handles
directory creation — the caller just specifies a directory and gets files.

</details>